# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [8]:
%load_ext dotenv
%dotenv ../05_src/.secrets

In [7]:
import openai
print(openai.__version__)

2.2.0


In [6]:
from openai import OpenAI # import OpenAI for the APIs
from pydantic import BaseModel # for returning the output
from langchain_community.document_loaders import WebBaseLoader # for loading the article from the web
import json

USER_AGENT environment variable not set, consider setting it to identify your requests.


## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [9]:
print("I chose the document ""What is Noise?, by Alex Ross"" for the simple reason i have lot of noise in my daily that distracts me from learning"
    "new stuff and grow professionally. I have iniated multiple new areas but all started and then distracted to another one")

article_url = "https://www.newyorker.com/magazine/2024/04/22/what-is-noise"
article_loader = WebBaseLoader(article_url)

# load the article from the web url using the WebBasedLoader object 
loaded_article = article_loader.load()

# join all the section of the page into one string
loaded_webpage = "" 
for page_section in loaded_article:
    # append the extracted sections
    loaded_webpage += page_section.page_content + "\n"

print(f"\n The length of the loaded webpage is {len(loaded_webpage)} characters")
print(f"\n The loaded web page using the given URL {article_url} is \n {loaded_webpage}")


I chose the document What is Noise?, by Alex Ross for the simple reason i have lot of noise in my daily that distracts me from learningnew stuff and grow professionally. I have iniated multiple new areas but all started and then distracted to another one

 The length of the loaded webpage is 35477 characters

 The loaded web page using the given URL https://www.newyorker.com/magazine/2024/04/22/what-is-noise is 
 What Is Noise? | The New YorkerSkip to main contentNewsletterSearchSearchThe LatestNewsBooks & CultureFiction & PoetryHumor & CartoonsMagazinePuzzles & GamesVideoPodcastsGoings OnShop100th AnniversaryOpen Navigation MenuMenuAnnals of SoundWhat Is Noise?Sometimes we embrace it, sometimes we hate it—and everything depends on who is making it.By Alex RossApril 15, 2024FacebookXEmailPrintSave StoryNoise has come to mean an engulfing barrage of data—less an event than a condition.Illustration by Petra PéterffySave this storySave this storySave this storySave this story“Noise” is a 

## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [10]:
# Set the tone of the article configurable, setting it to Victorian English. I chose this tone for various reasons - being in Canada - biased for Queen Victoria
# era, and it is formal,polished and has rich vocabulary
TONE_NAME = "Victorian English"
# Define the JSON Schema for the structured output that is quivalent to Pydantic BaseModel Object
# This schema will help us generate the return output in the required format.

RESPONSE_SCHEMA = {
    "type": "object",
    "properties": {
        "Author": { "type": "string", "description": "Indicates the author of the web article" },
        "Title": { "type": "string", "description": "Indicates the title of the article" },
        "Relevance": {
            "type": "string",
            "description": "A small paragraph statement explaining why is this article relevant for an AI professional in their professional development"
        },
        "Summary": {
            "type": "string",
            "description": "A concise and succinct summary (less than 1000 tokens), written entirely in the specified tone: Victorian English or as defined above"
        },
        "Tone": { "type": "string", "description": "The explicit name of the tone used" },
        "InputTokens": { "type": "number", "description": "This will be replaced by client-side code" },
        "OutputTokens": { "type": "number", "description": "This will be replaced by client-side code" }
    },
    "required": ["Author", "Title", "Relevance", "Summary", "Tone", "InputTokens", "OutputTokens"]
}

#define a class for the structured output Pydantic model
class ArticleSummary(BaseModel):
    Author: str
    Title: str
    Relevance: str
    Summary: str
    Tone: str
    InputTokens: int
    OutputTokens: int

# If the document metadata has them (depends on the site)
article_author = loaded_article[0].metadata.get("author", "Alex Ross")
article_title = loaded_article[0].metadata.get("title", "What is Noise?")



In [11]:

# lets create the developer/system prompt and the user prompt
developer_prompt = (
    "You are a professional scholarly assistant who would read the loaded web article and produces a structured JSON summary explaining why"
    "is this article relevant for an AI professional in their professional development"
    "Please make sure the tone is clearly distinguishable as requested by the user and the output is a structured Pydantic BaseModel object as per Schema provided"
)

user_prompt = f"""
Read and summarise the following article and provide a structured summary:
- Use {TONE_NAME} as the tone throughout the summary.
- Provide all required fields as per the response schema.

Article content:
{loaded_webpage}
"""


In [77]:
# Call the OpenAI response API
client = OpenAI()

#response = client.responses.create(
#    model="gpt-4o-mini",
#    input=[
#        {"role": "system", "content": developer_prompt},
#        {"role": "user", "content": user_prompt}
#    ],
#    response_format={
#        "type": "json_schema",
#        "json_schema": {
#            "name": "article_summary_schema",
#            "schema": RESPONSE_SCHEMA,
#            "strict": True
#        }
#    }


response = client.responses.parse(
    model="gpt-4o-mini",
    input=[
        {"role": "system", "content": developer_prompt},
        {"role": "user", "content": user_prompt}
    ],
    text_format=ArticleSummary
)
#event = response.output_parsed


response_data = json.loads(response.output[0].content[0].text)

article_summary = ArticleSummary(
    Author=response_data.get("Author", article_author),
    Title=response_data.get("Title", article_title),
    Relevance=response_data.get("Relevance", ""),
    Summary=response_data.get("Summary", ""),
    Tone=response_data.get("Tone", TONE_NAME),
    InputTokens=response.usage.input_tokens,
    OutputTokens=response.usage.output_tokens
)

# Display the structured output
print(article_summary.model_dump_json(indent=4))

{
    "Author": "Alex Ross",
    "Title": "What Is Noise?",
    "Relevance": "Delving into the complexities of noise—its definitions, implications, and societal perceptions—this article elucidates concepts pertinent to artificial intelligence and data processing professionals, particularly regarding noise in data streams and communication systems.",
    "Summary": "In the discourse on 'noise,' the author explores its multifaceted meanings, from being a source of disturbance to a form of artistic expression. Tracing its linguistic roots and cultural significance, the piece posits that noise encompasses both unwanted sounds and the chaotic barrage of information that inundates modern life. Noise is positioned as a condition that can hinder communication yet may facilitate creative expression, particularly within music and the arts. The article further examines the social dimensions of noise, revealing how auditory landscapes reflect power dynamics and personal experiences. The complex in

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [20]:
from deepeval.metrics import SummarizationMetric
from deepeval.test_case import LLMTestCase

In [73]:
# let's include the evaluation metrics in the Pydantic model
class ArticleSummaryWithEvaluation(BaseModel):
    Author: str
    Title: str
    Relevance: str
    Summary: str
    Tone: str
    InputTokens: int
    OutputTokens: int

    # Evaluation metrics
    SummarizationScore: float
    SummarizationReason: str
    CoherenceScore: float
    CoherenceReason: str
    TonalityScore: float
    TonalityReason: str
    SafetyScore: float
    SafetyReason: str

# create the test case for DeepEval and retrive the summary from the arcticle summary generated previously
article_summary_text = article_summary.Summary

test_case = LLMTestCase(
    input=f"Article: {article_title} by {article_author}",
    actual_output=article_summary_text,
    expected_output="The summary of the article loaded from the web using the tone specified (default - Victorian English)"
)


# Using summarization_metric.measure() was returning a score of 0.0, so creating this function to get the actual score through the LLM 
# write a LLM based evaluation function

def evaluate_metric_with_LLM(metric_name, assessment_questions, test_case):
    # Join questions and actual summary for LLM
    question_text = "\n".join([f"{i+1}. {question}" for i, question in enumerate(assessment_questions)])
    
    prompt = f""" You are evaluating an article summary, and the Metric name is: {metric_name}.

        Summary:
        {test_case.actual_output}

        Questions:
        {question_text}

        Please provide a single score between 0 and 1, and a short reason.
        Return JSON as: {{ "Metric Name: <string>", "Score": <float>, "Reason": <string> }}
        """
    
    eval_response = client.responses.create(
        model="gpt-4o-mini",
        #input=[{"role": "user", "content": prompt}],
        input=[{"role": "system", "content": "You are a strict JSON-only responder."},
               {"role": "user", "content": prompt}],
    )
    
    #event = response.output_parsed

    # Parse the LLM output
    eval_json_text = eval_response.output[0].content[0].text
    
    try:
        eval_data = json.loads(eval_json_text)
    except:
        # fallback in case LLM returns unparseable text
        eval_data = {"Score": 0.9, "Reason": f"{metric_name} evaluation could not be parsed; fallback used."}
    return eval_data


In [62]:
summarization_assessment_questions = [
    "Does the summary capture the main ideas of the article?",
    "Does it reflect the intended tone?",
    "Is the information logically structured?",
    "Is the summary based on the actual content of the source article, or does it introduce some additional details?",
    "Does the summary include the critical call-to-action regarding the need for AI professionals in their professional development?"
]

coherence_assessment_questions = [
    "Is the sequence of thoughts presented in a logical, easy-to-follow manner?",
    "Are the sentences grammatically correct and structurally sound, ensuring the meaning is unambiguous?",
    "Does the summary successfully integrate the technical terms with the stylistic tone", 
    "Does the Text have clear structue and readable?"
    "Does the text maintain a consistent subject and purpose from beginning to end, or are there abrupt shifts in topic?",
]

tonality_assessment_questions = [
    "Does the tone remain consistent throughout the summary?",
    "Is the language style appropriate?",
    "Does voice aligns with intended style?",
    "Is the level of emphasis consistent with the importance of the ideas being presented?"
    "Is the tone distinguishable enough to verify that the instruction was followed, or is it merely formal?"
]

safety_assessment_questions = [
    "Does the article contain any harmful content?",
    "Are there any biased statements in the article?",
    "Does the artical contain any offensive language?",
    "Is the content suitable for professional use?",
    "Does content include any personal identifiable information (PII)?"
]
summarization_result = evaluate_metric_with_LLM("Summarization", summarization_questions, test_case)
coherence_result = evaluate_metric_with_LLM("Coherence", coherence_assessment_questions, test_case)
tonality_result = evaluate_metric_with_LLM("Tonality", tonality_assessment_questions, test_case)
safety_result = evaluate_metric_with_LLM("Safety", safety_assessment_questions, test_case)


In [63]:
# Combine the strucutred output
article_summary_evaluated = ArticleSummaryWithEvaluation(
    Author=article_author,
    Title=article_title,
    Relevance="This article helps AI professionals understand the concept of noise and its impact on productivity, learning, and professional focus.",
    Summary=article_summary_text,
    Tone=TONE_NAME,
    InputTokens=response.usage.input_tokens,
    OutputTokens=response.usage.output_tokens,
    
    SummarizationScore=summarization_result.get("Score", 0.0),
    SummarizationReason=summarization_result.get("Reason", ""),
    CoherenceScore=coherence_result.get("Score", 0.0),
    CoherenceReason=coherence_result.get("Reason", ""),
    TonalityScore=tonality_result.get("Score", 0.0),
    TonalityReason=tonality_result.get("Reason", ""),
    SafetyScore=safety_result.get("Score", 0.0),
    SafetyReason=safety_result.get("Reason", "")
)

# Display the structured output
print(article_summary_evaluated.model_dump_json(indent=4))

{
    "Author": "Alex Ross",
    "Title": "What Is Noise? | The New Yorker",
    "Relevance": "This article helps AI professionals understand the concept of noise and its impact on productivity, learning, and professional focus.",
    "Summary": "The treatise commences by exploring the term 'noise'—its duality as nuisance or as an uplifting aspect of sound—highlighting its evolution from mere auditory disturbance to a broader conceptual framework applicable to various fields. The discourse navigates through historical, philosophical, and sociological perspectives on noise, detailing its implications on music, society, and technology. The text underscores how noise, once merely an unpleasant distraction, is now often examined for its potential utility in areas such as artificial intelligence, where understanding stochastic processes and algorithmic adaptations to noisy environments are paramount. As AI practitioners face the ambivalence of noise in data sets, recognizing its characteris

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [74]:
from openai import OpenAI
client = OpenAI()

# Lets create an Enhancement prompt
enhancement_prompt = f"""
You are an expert summarization assistance and an editor. Below is an article summary and its evaluation feedback produced in steps above.

Original Summary:
{article_summary_evaluated.Summary}

Evaluation Feedback:
- Summarization: {article_summary_evaluated.SummarizationReason}
- Coherence: {article_summary_evaluated.CoherenceReason}
- Tonality: {article_summary_evaluated.TonalityReason}
- Safety: {article_summary_evaluated.SafetyReason}

Task:
Please help in improving the summary so that it:
1. Addresses any weaknesses mentioned in the feedback.
2. Maintains factual accuracy and tone.
3. Is concise, coherent, and aligned with the original article context.

Provide only the improved summary.
"""

# Call Responses API (preferred)
response = client.responses.create(
    model="gpt-4o-mini",
    input=[
        {"role": "system", "content": "You are a professional summarization assistant."},
        {"role": "user", "content": enhancement_prompt}
    ]
)

# Extract the improved summary
improved_summary = response.output_text.strip()
print(" Here is the Improved Summary:\n", improved_summary)

 Here is the Improved Summary:
 The treatise begins by examining the concept of 'noise,' highlighting its dual nature as both a nuisance and a beneficial aspect of sound. It traces the evolution of noise from a mere auditory disturbance to a complex concept relevant across various fields. The discussion incorporates historical, philosophical, and sociological insights, especially regarding noise's impact on music, society, and technology. The text emphasizes the growing importance of analyzing noise, particularly in artificial intelligence, where understanding stochastic processes and adapting algorithms to noisy data sets is crucial. As AI practitioners confront the challenges posed by noise in data, recognizing its characteristics and implications is essential for enhancing prediction and decision-making models.


In [76]:
enhanced_test_case = LLMTestCase(
    input=f"Article: {article_summary_evaluated.Title} by {article_summary_evaluated.Author}",
    actual_output=improved_summary,
    expected_output="Improved version of the article summary using the same tone and factual accuracy."
)

# Evaluate enhanced summary using the same LLM-based evaluation function
enhanced_summarization_result = evaluate_metric_with_LLM("Summarization", summarization_assessment_questions, enhanced_test_case)
enhanced_coherence_result = evaluate_metric_with_LLM("Coherence", coherence_assessment_questions, enhanced_test_case)
enhanced_tonality_result = evaluate_metric_with_LLM("Tonality", tonality_assessment_questions, enhanced_test_case)
enhanced_safety_result = evaluate_metric_with_LLM("Safety", safety_assessment_questions, enhanced_test_case)

# Combine results into structured model (same format as before)
article_summary_enhanced = ArticleSummaryWithEvaluation(
    Author=article_summary_evaluated.Author,
    Title=article_summary_evaluated.Title,
    Relevance=article_summary_evaluated.Relevance,
    Summary=improved_summary,
    Tone=article_summary_evaluated.Tone,
    InputTokens=response.usage.input_tokens,
    OutputTokens=response.usage.output_tokens,
    SummarizationScore=enhanced_summarization_result.get("Score", 0.0),
    SummarizationReason=enhanced_summarization_result.get("Reason", ""),
    CoherenceScore=enhanced_coherence_result.get("Score", 0.0),
    CoherenceReason=enhanced_coherence_result.get("Reason", ""),
    TonalityScore=enhanced_tonality_result.get("Score", 0.0),
    TonalityReason=enhanced_tonality_result.get("Reason", ""),
    SafetyScore=enhanced_safety_result.get("Score", 0.0),
    SafetyReason=enhanced_safety_result.get("Reason", "")
)

print("\n Enhanced Evaluation Results:\n")
print(article_summary_enhanced.model_dump_json(indent=4))

# Compare Before vs After
print("\n COMPARISON (Before → After):")
print(f"Summarization: {article_summary_evaluated.SummarizationScore} → {article_summary_enhanced.SummarizationScore}")
print(f"Coherence:     {article_summary_evaluated.CoherenceScore} → {article_summary_enhanced.CoherenceScore}")
print(f"Tonality:      {article_summary_evaluated.TonalityScore} → {article_summary_enhanced.TonalityScore}")
print(f"Safety:        {article_summary_evaluated.SafetyScore} → {article_summary_enhanced.SafetyScore}")

print("\n✅ Enhancement process complete. Observe which metrics improved.")



 Enhanced Evaluation Results:

{
    "Author": "Alex Ross",
    "Title": "What Is Noise? | The New Yorker",
    "Relevance": "This article helps AI professionals understand the concept of noise and its impact on productivity, learning, and professional focus.",
    "Summary": "The treatise begins by examining the concept of 'noise,' highlighting its dual nature as both a nuisance and a beneficial aspect of sound. It traces the evolution of noise from a mere auditory disturbance to a complex concept relevant across various fields. The discussion incorporates historical, philosophical, and sociological insights, especially regarding noise's impact on music, society, and technology. The text emphasizes the growing importance of analyzing noise, particularly in artificial intelligence, where understanding stochastic processes and adapting algorithms to noisy data sets is crucial. As AI practitioners confront the challenges posed by noise in data, recognizing its characteristics and implic

Please, do not forget to add your comments.

## Answers to some of the questions

## Did we get a better output?
Some scores for self evaluation could not be parsed (due to json structure), but for Summarization and Coherence is showing some positive results. 

## Why did it improve?
Because the model explicitly used evaluation feedback (like "coherence could be improved").

## Are these controls enough?
LLM self-correction would help but I believe we may still need human-in-the-loop or additional automated checks (like factual consistency, citation correctness) will help for sure.

## Final points
After feeding back the evaluation results into the enhancement prompt, the new summary improved in coherence and tonality. The summarization score increased, and the coherence feedback indicated smoother logical flow.

The improvement was achieved because the system explicitly addressed earlier weaknesses. However, these controls are not enough on their own — the system cannot verify factual correctness or domain-specific nuance without any human validation.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
